In [ ]:
# importing libs
import zipfile
import shutil
from glob import glob

# mounting to Google Drive
from google.colab import drive
drive.mount('/content/drive')

# paths
zip1_path = "/content/drive/MyDrive/archive.zip"
zip2_path = "/content/drive/MyDrive/archive2.zip"

# temp extraction files
tmp1 = "/content/archive1_tmp"
tmp2 = "/content/archive2_tmp"
os.makedirs(tmp1, exist_ok=True)
os.makedirs(tmp2, exist_ok=True)

# extracting archives
# (the thrip images)
with zipfile.ZipFile(zip1_path, 'r') as zip_ref:
    zip_ref.extractall(tmp1)

with zipfile.ZipFile(zip2_path, 'r') as zip_ref:
    zip_ref.extractall(tmp2)

# folder for thrips dataset
thrips_dataset_path = "/content/drive/MyDrive/thrips_dataset"
os.makedirs(thrips_dataset_path, exist_ok=True)

# cpoying all images
def copy_thrips_images(src_folder, dest_folder):
    for root, dirs, files in os.walk(src_folder):
        if "Thrips" in root:
            for file in files:
                if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                    shutil.copy(os.path.join(root, file), dest_folder)

# from both archives
copy_thrips_images(tmp1, thrips_dataset_path)
copy_thrips_images(tmp2, thrips_dataset_path)

# renaming files for consistency
all_images = sorted(glob(os.path.join(thrips_dataset_path, "*.*")))
for idx, filepath in enumerate(all_images, start=1):
    ext = os.path.splitext(filepath)[1].lower()
    new_name = f"thrips_{idx:03d}{ext}"
    new_path = os.path.join(thrips_dataset_path, new_name)
    os.rename(filepath, new_path)

print(f"Done! Consolidated {len(all_images)} Thrips images into '{thrips_dataset_path}'.")


In [ ]:
import pandas as pd
import requests
import os
import json

# more paths, this is the data downloaded from zooniverse
csv_file = "/content/drive/MyDrive/rhs-wisley-bug-watch-subjects.csv"
download_folder = "/content/drive/MyDrive/test_set"
os.makedirs(download_folder, exist_ok=True)

# Load CSV
df = pd.read_csv(csv_file)

count = 0
for idx, row in df.iterrows():
    # parsing the JSON string in 'locations' column
    try:
        loc_dict = json.loads(row['locations'])
        for key, url in loc_dict.items():
            response = requests.get(url, stream=True)
            if response.status_code == 200:
              # handling ?
                ext = url.split('.')[-1].split('?')[0]
                filepath = os.path.join(download_folder, f"test_{count:04d}.{ext}")
                with open(filepath, 'wb') as f:
                    f.write(response.content)
                count += 1
            else:
                print(f"Failed to download {url}")
    except Exception as e:
        print(f"Error parsing row {idx}: {e}")

print(f"Downloaded {count} images to {download_folder}")


Aggregation

Just a quick note: i only did majority-vote aggregation, considered doing like an averaged one instead it'll be interestin to see if that changes anything

In [ ]:
# DID A MOJIRTY-VOTE AGGREGATION

import pandas as pd
import json

csv_path = "/content/drive/MyDrive/rhs-wisley-bug-watch-classifications.csv"
df = pd.read_csv(csv_path)

records = []

for _, row in df.iterrows():
    annotations = json.loads(row["annotations"])
    subject_data = json.loads(row["subject_data"])

    subject_id = str(row["subject_ids"])
    subject_entry = subject_data[subject_id]


    filename = None


    if "Filename" in subject_entry:
        filename = subject_entry["Filename"]


    elif "filename" in subject_entry:
        filename = subject_entry["filename"]

    # extracting locations from url
    elif "locations" in subject_entry:
        url = list(subject_entry["locations"][0].values())[0]
        filename = os.path.basename(url.split("?")[0])

    else:
        continue


    for ann in annotations:
        if ann["task"] == "T1":
            records.append({
                "subject_id": subject_id,
                "filename": filename,
                "vote": ann["value"]
            })

votes_df = pd.DataFrame(records)
votes_df.head()

In [ ]:
agg = (
    votes_df
    .groupby(["subject_id", "filename"])["vote"]
    .value_counts()
    .unstack(fill_value=0)
    .reset_index()
)

agg["total_votes"] = agg.sum(axis=1, numeric_only=True)

agg["label"] = agg.apply(
    lambda r: "Thrips" if r.get("Yes", 0) > r.get("No", 0) else "Other",
    axis=1
)


agg = agg[agg["total_votes"] >= 3]

agg.head()


In [ ]:
agg["label"].value_counts()


In [ ]:
# using a resnet classifier
import os
import shutil

# test folder
raw_images = "/content/drive/MyDrive/test_set"
test_root = "/content/test"

os.makedirs(f"{test_root}/Thrips", exist_ok=True)
os.makedirs(f"{test_root}/Other", exist_ok=True)

files = os.listdir(raw_images)

thrips_count = 0
other_count = 0

for i, (_, row) in enumerate(agg.iterrows()):
    if i >= len(files):
        break

    src = os.path.join(raw_images, files[i])

    if row["label"] == "Thrips":
        dst = os.path.join(test_root, "Thrips", files[i])
        thrips_count += 1
    else:
        dst = os.path.join(test_root, "Other", files[i])
        other_count += 1

    shutil.copy(src, dst)

print("Thrips:", thrips_count)
print("Other:", other_count)


print("Test set prepared!")

# temp fallback
train_root = "/content/train_quick"
os.makedirs(train_root, exist_ok=True)

# temp baseline nonthrips
os.makedirs(f"{train_root}/Thrips", exist_ok=True)
os.makedirs(f"{train_root}/Other", exist_ok=True)

# copying the thrips
for f in os.listdir("/content/thrips_dataset"):
    shutil.copy(f"/content/thrips_dataset/{f}", f"{train_root}/Thrips")

# temp fallback
for f in os.listdir(f"{test_root}/Other")[:200]:
    shutil.copy(f"{test_root}/Other/{f}", f"{train_root}/Other")

print("Train set prepared!")

# modelling
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

train_ds = datasets.ImageFolder(train_root, transform=transform)
test_ds  = datasets.ImageFolder(test_root, transform=transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 2)
model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# acc training thr model
for epoch in range(2):
    model.train()
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1} done")




# EVAL
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

model.eval()
preds, labels = [], []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        out = model(x)
        preds.extend(out.argmax(1).cpu().numpy())
        labels.extend(y.numpy())

print(classification_report(labels, preds, target_names=test_ds.classes))

cm = confusion_matrix(labels, preds)
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=test_ds.classes,
            yticklabels=test_ds.classes)
plt.show()


Performance is absolutely awful like rlly bad and everything classified as "Other"           
Refined Approach to address class imbalance + modified decision thresholds

In [ ]:


# trying to address class imbalance + changed thresholds

import os
import shutil
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# rebuilt
train_root = "/content/train_balanced"

# reset folder
if os.path.exists(train_root):
    shutil.rmtree(train_root)

os.makedirs(f"{train_root}/Thrips", exist_ok=True)
os.makedirs(f"{train_root}/Other", exist_ok=True)

thrips_src = "/content/thrips_dataset"
other_src  = "/content/test/Other"

thrips_files = os.listdir(thrips_src)
other_files  = os.listdir(other_src)

# balancing classes
n = min(len(thrips_files), len(other_files))

thrips_sample = random.sample(thrips_files, n)
other_sample  = random.sample(other_files, n)

# copy balanced data
for f in thrips_sample:
    shutil.copy(os.path.join(thrips_src, f), f"{train_root}/Thrips")

for f in other_sample:
    shutil.copy(os.path.join(other_src, f), f"{train_root}/Other")

print(f"Balanced training set: {n} Thrips, {n} Other")

# using augmentation to transform
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor()
])

# load datasets
train_ds = datasets.ImageFolder(train_root, transform=transform)
test_ds  = datasets.ImageFolder("/content/test", transform=transform)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False)

# modelling
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

# class weighting
class_counts = torch.tensor([
    len([x for x in train_ds.samples if x[1] == 0]),  # Other
    len([x for x in train_ds.samples if x[1] == 1])   # Thrips
], dtype=torch.float)

weights = 1.0 / class_counts
weights = weights / weights.sum()

criterion = nn.CrossEntropyLoss(weight=weights.to(device))
optimizer = optim.Adam(model.parameters(), lr=1e-4)

#acc training
EPOCHS = 5

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss/len(train_loader):.4f}")

#EVAL
model.eval()
preds, labels = [], []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        out = model(x)

        probs = torch.softmax(out, dim=1)

        # using a lower threshold to improve detection
        thrips_preds = (probs[:,1] > 0.3).int().cpu().numpy()

        preds.extend(thrips_preds)
        labels.extend(y.numpy())

# new results
print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(labels, preds, target_names=test_ds.classes))

cm = confusion_matrix(labels, preds)

plt.figure(figsize=(4,4))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=test_ds.classes,
            yticklabels=test_ds.classes)
plt.title("Improved Model Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


In [ ]:
model.eval()
preds, labels = [], []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        out = model(x)

        probs = torch.softmax(out, dim=1)

        # superrrr low threshold to force literally any kind of detection atp
        thrips_preds = (probs[:,1] > 0.1).int().cpu().numpy()

        preds.extend(thrips_preds)
        labels.extend(y.numpy())


print("Sample probabilities:")
print(probs[:10])
print(test_ds.classes)



DECIDED ON A COMPLETELY NEW APPROACH:
- oversample thrips using weightedRandomSampler (so model sees thrip images more often)
- weighted loss
- low threshold eval

In [ ]:

import torch
from torch.utils.data import DataLoader, WeightedRandomSampler
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

# dataset n transforms
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor()
])

train_ds = datasets.ImageFolder("/content/train_balanced", transform=transform)
test_ds  = datasets.ImageFolder("/content/test", transform=transform)

# using a weighted sampler to oversample thrips
# so by oversampling it basically forces the model to see more thrips
# bc increased exposure means increased change of detection (... well hopefully)
labels = [label for _, label in train_ds.samples]
class_sample_count = np.array([len([l for l in labels if l==i]) for i in range(2)])
weight_per_class = 1.0 / class_sample_count
samples_weight = np.array([weight_per_class[t] for t in labels])
samples_weight = torch.from_numpy(samples_weight).float()
sampler = WeightedRandomSampler(samples_weight, len(samples_weight), replacement=True)

train_loader = DataLoader(train_ds, batch_size=32, sampler=sampler)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False)

# model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

# incorporating weighted loss
class_counts = np.array([len([l for l in labels if l==i]) for i in range(2)])
weights = torch.tensor([class_counts[1]/class_counts[0], 1.0], dtype=torch.float).to(device)  # boost Thrips
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

#training
EPOCHS = 5
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss/len(train_loader):.4f}")

#EVAL w lower threshold
model.eval()
preds, labels_list = [], []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        out = model(x)
        probs = torch.softmax(out, dim=1)
        # rlllly low threshold
        thrips_preds = (probs[:,1] > 0.1).int().cpu().numpy()
        preds.extend(thrips_preds)
        labels_list.extend(y.numpy())

print("\nSample probabilities:")
print(probs[:10])
print(test_ds.classes)

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(labels_list, preds, target_names=test_ds.classes))

cm = confusion_matrix(labels_list, preds)
plt.figure(figsize=(4,4))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=test_ds.classes,
            yticklabels=test_ds.classes)
plt.title("Balanced + Oversampled Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


APPROACH NO. 4:
- extreme oversampling
- stronger augmentation aka flips rotations
- freeze pretrained layers so the final classifier is only fine tuned


- thrips oversampled 10x so model sees them more frequently
- still using weighted loss
-low threshold eval at >0.1

In [ ]:

import torch
from torch.utils.data import DataLoader, ConcatDataset
from torchvision import datasets, transforms, models
import torch.nn as nn
import torch.optim as optim
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import random
import shutil
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# STRONGER AUG
train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomResizedCrop(224, scale=(0.7,1.0)),
    transforms.ToTensor()
])

test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

# loading datasetds
train_root = "train_balanced"
train_ds = datasets.ImageFolder(train_root, transform=train_transform)
test_ds  = datasets.ImageFolder("test", transform=test_transform)

# extreme oversampling
insect_class_id = train_ds.class_to_idx['Insect']
other_class_id = train_ds.class_to_idx['Other']

insect_idx = [i for i, (_, label) in enumerate(train_ds.samples) if label == insect_class_id]
other_idx  = [i for i, (_, label) in enumerate(train_ds.samples) if label == other_class_id]

# repeat insect 10x
insect_datasets = [torch.utils.data.Subset(train_ds, insect_idx)] * 10
other_datasets = [torch.utils.data.Subset(train_ds, other_idx)]
# Combine with Other once
balanced_ds = ConcatDataset([*insect_datasets, *other_datasets])
train_loader = DataLoader(balanced_ds, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False)

# freeze pretrained layersd
model = models.resnet18(pretrained=True)
for name, param in model.named_parameters():
    if "fc" not in name:
        param.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

# weighted loss
class_counts = np.array([len(other_idx), len(insect_idx)])
weights = torch.tensor([1.0, 1.0], dtype=torch.float)
weights[insect_class_id] = class_counts[0] / max(1, class_counts[1]) 
weights[other_class_id] = 1.0
weights = weights.to(device)
criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = optim.Adam(model.fc.parameters(), lr=1e-4)  # only fc parameters train

# training
EPOCHS = 15
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {total_loss/len(train_loader):.4f}")

#eval w low threshold
model.eval()
preds, labels_list, all_probs = [], [], []

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        out = model(x)
        probs = torch.softmax(out, dim=1)
        insect_preds = (probs[:, insect_class_id] > 0.5).int().cpu().numpy()
        preds.extend(insect_preds)
        labels_list.extend(y.numpy())
        all_probs.append(probs.cpu())

all_probs = torch.cat(all_probs)
print("\nSample probabilities:")
print(all_probs[:10])
print(test_ds.classes)


print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(labels_list, preds, target_names=test_ds.classes))

cm = confusion_matrix(labels_list, preds)
plt.figure(figsize=(4,4))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=test_ds.classes,
            yticklabels=test_ds.classes)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()
